<a href="https://colab.research.google.com/github/SANGHATI23/neurofhir-qc/blob/main/15_NeuroFHIR_Review_Reviewer_Training_Rubric.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# Cell 1 — Setup

from pathlib import Path
import json, csv, hashlib, datetime, textwrap, html
from IPython.display import display, Markdown, HTML

# Keep WISH artifacts separate from the frozen AMIA FHIR App build.
BASE_DIR = Path("/content/neurofhir_review_wish")
OUT_DIR = BASE_DIR / "step08_reviewer_training"
OUT_DIR.mkdir(parents=True, exist_ok=True)

VERSION = "1.0"
STUDY_NAME = "NeuroFHIR-Review WISH 2026"

print("Output directory:", OUT_DIR)
print("Training version:", VERSION)


Output directory: /content/neurofhir_review_wish/step08_reviewer_training
Training version: 1.0



## Reviewer-facing logic used in this notebook

The training preserves the app's existing reconciliation actions:

- **Accept AI**
- **Keep initial judgment**
- **Amend**
- **Reject**
- **Escalate**

The reviewer should not be told which study cases contain deliberate conflicts or which answer is expected.  
Training therefore teaches **general decision rules only** and does not reveal the 12-case matrix.

For Path B, the key scientific endpoints remain workflow reasoning, uncertainty/provenance use, escalation behavior, confidence calibration, and usability — not diagnostic accuracy.


In [2]:

# Cell 2 — Canonical reviewer-training specification

TRAINING_SPEC = {
    "study_name": STUDY_NAME,
    "version": VERSION,
    "task_statement": (
        "Use only the evidence displayed in the review interface to decide how the AI output "
        "should be handled in the workflow. This is a workflow-evaluation task, not a request "
        "to make a neurological diagnosis."
    ),
    "review_sequence": [
        "Inspect the current and prior measurements, longitudinal change, and visible QC/uncertainty information.",
        "Form the required independent initial judgment and confidence when the interface asks for it.",
        "After AI advice is visible, compare it against the displayed evidence rather than treating confidence as proof.",
        "Open provenance when evidence is conflicting, incomplete, surprising, or materially relevant to the decision.",
        "Submit the final reconciliation action, final confidence, and the shortest accurate reason code."
    ],
    "actions": {
        "Accept AI": (
            "Use when the AI recommendation is adequately supported by the displayed evidence, "
            "QC/uncertainty is acceptable for the case, and no material provenance or longitudinal conflict is visible."
        ),
        "Keep initial judgment": (
            "Use when the AI advice does not provide enough justified evidence to change the initial judgment, "
            "but the case does not require outright rejection or escalation."
        ),
        "Amend": (
            "Use when newly revealed AI-linked evidence legitimately changes part of the initial judgment. "
            "The amendment should be evidence-based, not confidence-based."
        ),
        "Reject": (
            "Use when the AI recommendation is materially contradicted by the displayed evidence or is not supportable "
            "from the evidence shown."
        ),
        "Escalate": (
            "Use when the available evidence is insufficient for a responsible final workflow decision, especially when "
            "QC is poor, provenance is materially incomplete, uncertainty is consequential, or signals remain discordant."
        ),
    },
    "confidence_scale": {
        "1": "Very low confidence — evidence is insufficient or highly conflicting.",
        "2": "Low confidence — important uncertainty remains.",
        "3": "Moderate confidence — evidence is mixed or only partly decisive.",
        "4": "High confidence — evidence is largely consistent with the decision.",
        "5": "Very high confidence — displayed evidence strongly and coherently supports the decision."
    },
    "reason_codes": [
        "Evidence supports AI",
        "AI conflicts with longitudinal evidence",
        "Low confidence / QC concern",
        "Missing or incomplete provenance",
        "Evidence insufficient",
        "AI evidence changed initial judgment",
        "Other"
    ],
    "reviewer_rules": [
        "Use the displayed evidence; do not invent missing clinical context.",
        "AI confidence is one signal, not proof that the recommendation is correct.",
        "A provenance gap matters when it prevents you from understanding or trusting how the evidence was produced.",
        "Low-confidence or QC-failed evidence should trigger additional inspection and may justify escalation.",
        "When AI and longitudinal evidence conflict, inspect the evidence chain before finalizing.",
        "Do not try to guess the study condition or hidden expected answer.",
        "Do not interpret this exercise as clinical validation or as a test of diagnostic competence."
    ],
    "analysis_note": (
        "Path B analyses should focus on workflow disposition, evidence/provenance inspection, escalation, "
        "confidence calibration, automation-bias proxies, usability, and reasoning themes."
    )
}

print(json.dumps(TRAINING_SPEC, indent=2))


{
  "study_name": "NeuroFHIR-Review WISH 2026",
  "version": "1.0",
  "task_statement": "Use only the evidence displayed in the review interface to decide how the AI output should be handled in the workflow. This is a workflow-evaluation task, not a request to make a neurological diagnosis.",
  "review_sequence": [
    "Inspect the current and prior measurements, longitudinal change, and visible QC/uncertainty information.",
    "Form the required independent initial judgment and confidence when the interface asks for it.",
    "After AI advice is visible, compare it against the displayed evidence rather than treating confidence as proof.",
    "Open provenance when evidence is conflicting, incomplete, surprising, or materially relevant to the decision.",
    "Submit the final reconciliation action, final confidence, and the shortest accurate reason code."
  ],
  "actions": {
    "Accept AI": "Use when the AI recommendation is adequately supported by the displayed evidence, QC/uncertai

In [3]:

# Cell 3 — Generate the one-page Markdown reviewer rubric

spec = TRAINING_SPEC

rubric_md = f"""# NeuroFHIR-Review — Reviewer Training & Decision Rubric
**WISH 2026 | Version {spec['version']} | Research prototype — not for clinical use**

## Your task
{spec['task_statement']}

## Review sequence
1. {spec['review_sequence'][0]}
2. {spec['review_sequence'][1]}
3. {spec['review_sequence'][2]}
4. {spec['review_sequence'][3]}
5. {spec['review_sequence'][4]}

## Final reconciliation actions
| Action | Use it when... |
|---|---|
| **Accept AI** | {spec['actions']['Accept AI']} |
| **Keep initial judgment** | {spec['actions']['Keep initial judgment']} |
| **Amend** | {spec['actions']['Amend']} |
| **Reject** | {spec['actions']['Reject']} |
| **Escalate** | {spec['actions']['Escalate']} |

## Confidence (1–5)
**1** = very low · **2** = low · **3** = moderate · **4** = high · **5** = very high.
Confidence should follow the **displayed evidence**, not the AI's confidence alone.

## Reason codes
{'; '.join(spec['reason_codes'])}

## Rules to remember
- Use only displayed evidence; do not invent missing clinical context.
- AI confidence is a signal, not proof.
- Inspect provenance when evidence is conflicting, incomplete, surprising, or materially relevant.
- Low-confidence/QC-failed evidence may require escalation.
- Do not guess hidden study answers or case types.
- This is a **workflow/HCI evaluation**, not clinical validation or a diagnostic-accuracy test.
"""

RUBRIC_MD_PATH = OUT_DIR / "NeuroFHIR_Review_Reviewer_Rubric_v1.md"
RUBRIC_MD_PATH.write_text(rubric_md, encoding="utf-8")

display(Markdown(rubric_md))
print("\nSaved:", RUBRIC_MD_PATH)


# NeuroFHIR-Review — Reviewer Training & Decision Rubric
**WISH 2026 | Version 1.0 | Research prototype — not for clinical use**

## Your task
Use only the evidence displayed in the review interface to decide how the AI output should be handled in the workflow. This is a workflow-evaluation task, not a request to make a neurological diagnosis.

## Review sequence
1. Inspect the current and prior measurements, longitudinal change, and visible QC/uncertainty information.
2. Form the required independent initial judgment and confidence when the interface asks for it.
3. After AI advice is visible, compare it against the displayed evidence rather than treating confidence as proof.
4. Open provenance when evidence is conflicting, incomplete, surprising, or materially relevant to the decision.
5. Submit the final reconciliation action, final confidence, and the shortest accurate reason code.

## Final reconciliation actions
| Action | Use it when... |
|---|---|
| **Accept AI** | Use when the AI recommendation is adequately supported by the displayed evidence, QC/uncertainty is acceptable for the case, and no material provenance or longitudinal conflict is visible. |
| **Keep initial judgment** | Use when the AI advice does not provide enough justified evidence to change the initial judgment, but the case does not require outright rejection or escalation. |
| **Amend** | Use when newly revealed AI-linked evidence legitimately changes part of the initial judgment. The amendment should be evidence-based, not confidence-based. |
| **Reject** | Use when the AI recommendation is materially contradicted by the displayed evidence or is not supportable from the evidence shown. |
| **Escalate** | Use when the available evidence is insufficient for a responsible final workflow decision, especially when QC is poor, provenance is materially incomplete, uncertainty is consequential, or signals remain discordant. |

## Confidence (1–5)
**1** = very low · **2** = low · **3** = moderate · **4** = high · **5** = very high.  
Confidence should follow the **displayed evidence**, not the AI's confidence alone.

## Reason codes
Evidence supports AI; AI conflicts with longitudinal evidence; Low confidence / QC concern; Missing or incomplete provenance; Evidence insufficient; AI evidence changed initial judgment; Other

## Rules to remember
- Use only displayed evidence; do not invent missing clinical context.
- AI confidence is a signal, not proof.
- Inspect provenance when evidence is conflicting, incomplete, surprising, or materially relevant.
- Low-confidence/QC-failed evidence may require escalation.
- Do not guess hidden study answers or case types.
- This is a **workflow/HCI evaluation**, not clinical validation or a diagnostic-accuracy test.



Saved: /content/neurofhir_review_wish/step08_reviewer_training/NeuroFHIR_Review_Reviewer_Rubric_v1.md


In [4]:

# Cell 4 — Generate a compact print-friendly HTML version

def esc(x):
    return html.escape(str(x))

action_rows = "\n".join(
    f"<tr><td><b>{esc(k)}</b></td><td>{esc(v)}</td></tr>"
    for k, v in spec["actions"].items()
)

reason_codes = " · ".join(esc(x) for x in spec["reason_codes"])

rubric_html = f"""<!doctype html>
<html>
<head>
<meta charset="utf-8">
<title>NeuroFHIR-Review Reviewer Rubric v{VERSION}</title>
<style>
@page {{ size: Letter; margin: 0.45in; }}
body {{
  font-family: Arial, Helvetica, sans-serif;
  max-width: 8in;
  margin: 0 auto;
  color: #111;
  font-size: 11px;
  line-height: 1.25;
}}
h1 {{ font-size: 20px; margin: 0 0 2px; }}
h2 {{ font-size: 13px; margin: 9px 0 3px; }}
.small {{ font-size: 9px; }}
.banner {{
  border: 1px solid #555;
  padding: 7px 9px;
  margin: 6px 0;
}}
table {{
  width: 100%;
  border-collapse: collapse;
  font-size: 9.5px;
}}
th, td {{
  border: 1px solid #888;
  padding: 4px 5px;
  vertical-align: top;
}}
ol, ul {{ margin: 3px 0 3px 18px; padding: 0; }}
li {{ margin: 1px 0; }}
.footer {{
  margin-top: 7px;
  padding-top: 5px;
  border-top: 1px solid #aaa;
  font-size: 9px;
}}
</style>
</head>
<body>
<h1>NeuroFHIR-Review — Reviewer Training & Decision Rubric</h1>
<div class="small"><b>WISH 2026 · Version {esc(VERSION)} · Research prototype — not for clinical use</b></div>

<div class="banner"><b>Your task:</b> {esc(spec['task_statement'])}</div>

<h2>Review sequence</h2>
<ol>
  {''.join(f'<li>{esc(x)}</li>' for x in spec['review_sequence'])}
</ol>

<h2>Final reconciliation actions</h2>
<table>
<tr><th style="width:24%">Action</th><th>Use it when...</th></tr>
{action_rows}
</table>

<h2>Confidence scale</h2>
<div><b>1</b> very low · <b>2</b> low · <b>3</b> moderate · <b>4</b> high · <b>5</b> very high.
Confidence should follow the <b>displayed evidence</b>, not AI confidence alone.</div>

<h2>Reason codes</h2>
<div>{reason_codes}</div>

<h2>Rules to remember</h2>
<ul>
<li>Use only displayed evidence; do not invent missing clinical context.</li>
<li>AI confidence is a signal, not proof.</li>
<li>Inspect provenance when evidence is conflicting, incomplete, surprising, or materially relevant.</li>
<li>Low-confidence/QC-failed evidence may require escalation.</li>
<li>Do not guess hidden study answers or case types.</li>
<li>This is a workflow/HCI evaluation, not clinical validation or a diagnostic-accuracy test.</li>
</ul>

<div class="footer">
Reviewers should receive the same rubric version before the pilot. Case-specific expected answers are intentionally not included.
</div>
</body>
</html>
"""

RUBRIC_HTML_PATH = OUT_DIR / "NeuroFHIR_Review_Reviewer_Rubric_v1.html"
RUBRIC_HTML_PATH.write_text(rubric_html, encoding="utf-8")

display(HTML(rubric_html))
print("Saved:", RUBRIC_HTML_PATH)


Action,Use it when...
Accept AI,"Use when the AI recommendation is adequately supported by the displayed evidence, QC/uncertainty is acceptable for the case, and no material provenance or longitudinal conflict is visible."
Keep initial judgment,"Use when the AI advice does not provide enough justified evidence to change the initial judgment, but the case does not require outright rejection or escalation."
Amend,"Use when newly revealed AI-linked evidence legitimately changes part of the initial judgment. The amendment should be evidence-based, not confidence-based."
Reject,Use when the AI recommendation is materially contradicted by the displayed evidence or is not supportable from the evidence shown.
Escalate,"Use when the available evidence is insufficient for a responsible final workflow decision, especially when QC is poor, provenance is materially incomplete, uncertainty is consequential, or signals remain discordant."


Saved: /content/neurofhir_review_wish/step08_reviewer_training/NeuroFHIR_Review_Reviewer_Rubric_v1.html


In [5]:

# Cell 5 — Save machine-readable decision rules

RULES_PATH = OUT_DIR / "reviewer_decision_rules_v1.json"
RULES_PATH.write_text(json.dumps(TRAINING_SPEC, indent=2), encoding="utf-8")

print("Saved:", RULES_PATH)


Saved: /content/neurofhir_review_wish/step08_reviewer_training/reviewer_decision_rules_v1.json



## Optional pre-pilot comprehension check

This is **not** an outcome measure and should not expose the 12 study cases.  
Its purpose is only to verify that a trained reviewer understands the standardized workflow rules before entering the real pilot.

A study protocol can define the passing rule before recruitment. Do not decide the threshold after seeing participant performance.


In [6]:

# Cell 6 — Generate a short comprehension check and separate answer key

quiz = [
    {
        "item_id": "T01",
        "question": "The AI is very confident, but the displayed QC signal is poor. What should drive your decision?",
        "option_A": "The AI confidence alone",
        "option_B": "The displayed evidence, QC/uncertainty, and provenance",
        "option_C": "A guess about the hidden case type",
        "correct": "B",
        "rationale": "AI confidence is a signal, not proof; poor QC requires evidence-based scrutiny."
    },
    {
        "item_id": "T02",
        "question": "The AI conclusion conflicts with the longitudinal pattern. What is the best next step?",
        "option_A": "Accept immediately because the AI saw the image",
        "option_B": "Inspect the conflicting evidence/provenance before final action",
        "option_C": "Ignore the longitudinal evidence",
        "correct": "B",
        "rationale": "Discordance should trigger evidence-chain inspection before finalization."
    },
    {
        "item_id": "T03",
        "question": "When should Escalate be considered?",
        "option_A": "When evidence is insufficient, materially discordant, QC-poor, or provenance is consequentially incomplete",
        "option_B": "Only when AI confidence is high",
        "option_C": "Whenever the reviewer wants to skip the case",
        "correct": "A",
        "rationale": "Escalation is for unresolved evidence or accountability problems, not convenience."
    },
    {
        "item_id": "T04",
        "question": "What does this study ask you to prove?",
        "option_A": "That you can diagnose neurological disease",
        "option_B": "That the AI is clinically validated",
        "option_C": "How AI evidence should be handled in the displayed workflow",
        "correct": "C",
        "rationale": "Path B is a formative human-AI workflow evaluation, not diagnostic validation."
    },
    {
        "item_id": "T05",
        "question": "If AI advice is revealed after your initial judgment, what should you do?",
        "option_A": "Automatically change your answer to match the AI",
        "option_B": "Compare AI-linked evidence with the evidence already reviewed and reconcile explicitly",
        "option_C": "Ignore all AI-linked evidence",
        "correct": "B",
        "rationale": "The study measures accountable reconciliation, not reflexive acceptance or reflexive rejection."
    }
]

QUIZ_PATH = OUT_DIR / "reviewer_training_quiz_v1.csv"
KEY_PATH = OUT_DIR / "reviewer_training_quiz_answer_key_v1.json"

with QUIZ_PATH.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["item_id", "question", "option_A", "option_B", "option_C"]
    )
    writer.writeheader()
    for q in quiz:
        writer.writerow({k: q[k] for k in writer.fieldnames})

answer_key = {
    q["item_id"]: {"correct": q["correct"], "rationale": q["rationale"]}
    for q in quiz
}
KEY_PATH.write_text(json.dumps(answer_key, indent=2), encoding="utf-8")

print("Saved:", QUIZ_PATH)
print("Saved:", KEY_PATH)


Saved: /content/neurofhir_review_wish/step08_reviewer_training/reviewer_training_quiz_v1.csv
Saved: /content/neurofhir_review_wish/step08_reviewer_training/reviewer_training_quiz_answer_key_v1.json


In [7]:

# Cell 7 — Validate Step 8 materials against the WISH design boundary

required_actions = {"Accept AI", "Keep initial judgment", "Amend", "Reject", "Escalate"}
required_reason_codes = {
    "Evidence supports AI",
    "AI conflicts with longitudinal evidence",
    "Low confidence / QC concern",
    "Missing or incomplete provenance",
    "Evidence insufficient"
}

all_text = (
    rubric_md + "\n" +
    rubric_html + "\n" +
    json.dumps(TRAINING_SPEC) + "\n" +
    json.dumps(quiz)
).lower()

checks = {
    "all_five_app_actions_present": required_actions == set(TRAINING_SPEC["actions"].keys()),
    "core_reason_codes_present": required_reason_codes.issubset(set(TRAINING_SPEC["reason_codes"])),
    "explicit_non_diagnostic_boundary": "not a request to make a neurological diagnosis" in all_text,
    "explicit_no_clinical_validation_boundary": "not clinical validation" in all_text,
    "provenance_rule_present": "provenance" in all_text,
    "uncertainty_or_qc_rule_present": ("uncertainty" in all_text and "qc" in all_text),
    "initial_then_ai_reconciliation_present": (
        "initial judgment" in all_text and "after ai advice is visible" in all_text
    ),
    "no_case_specific_answers_in_rubric": "case_id" not in rubric_md.lower()
}

validation = {
    "study": STUDY_NAME,
    "version": VERSION,
    "checks": checks,
    "passed": all(checks.values())
}

VALIDATION_PATH = OUT_DIR / "step08_training_validation_report.json"
VALIDATION_PATH.write_text(json.dumps(validation, indent=2), encoding="utf-8")

for k, v in checks.items():
    print(("PASS" if v else "FAIL"), "-", k)

print("\nFINAL:", "PASS" if validation["passed"] else "FAIL")
print("Saved:", VALIDATION_PATH)

assert validation["passed"], "Step 8 validation failed. Fix the failed check(s) before using the materials."


PASS - all_five_app_actions_present
PASS - core_reason_codes_present
PASS - explicit_non_diagnostic_boundary
PASS - explicit_no_clinical_validation_boundary
PASS - provenance_rule_present
PASS - uncertainty_or_qc_rule_present
PASS - initial_then_ai_reconciliation_present
PASS - no_case_specific_answers_in_rubric

FINAL: PASS
Saved: /content/neurofhir_review_wish/step08_reviewer_training/step08_training_validation_report.json


In [8]:

# Cell 8 — Freeze the training version with SHA-256 checksums

artifact_paths = [
    RUBRIC_MD_PATH,
    RUBRIC_HTML_PATH,
    RULES_PATH,
    QUIZ_PATH,
    KEY_PATH,
    VALIDATION_PATH
]

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

manifest = {
    "study": STUDY_NAME,
    "step": 8,
    "training_version": VERSION,
    "generated_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "artifacts": [
        {
            "file": p.name,
            "sha256": sha256_file(p),
            "bytes": p.stat().st_size
        }
        for p in artifact_paths
    ]
}

MANIFEST_PATH = OUT_DIR / "step08_training_manifest_v1.json"
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print(json.dumps(manifest, indent=2))


{
  "study": "NeuroFHIR-Review WISH 2026",
  "step": 8,
  "training_version": "1.0",
  "generated_utc": "2026-09-11T03:54:34.019919+00:00",
  "artifacts": [
    {
      "file": "NeuroFHIR_Review_Reviewer_Rubric_v1.md",
      "sha256": "c54f8dd570933430727d8ae3809b03a37e4377a44463a31dae5ea30613386b84",
      "bytes": 2780
    },
    {
      "file": "NeuroFHIR_Review_Reviewer_Rubric_v1.html",
      "sha256": "b6dfa8ffe30478dba55330272016d066cd64549dc11be545580588ab5250f842",
      "bytes": 4164
    },
    {
      "file": "reviewer_decision_rules_v1.json",
      "sha256": "cdf6d08eded57102b022190fcbed5a4f75b8d521afe6b74cb42cb8ad47fbcde6",
      "bytes": 3444
    },
    {
      "file": "reviewer_training_quiz_v1.csv",
      "sha256": "8c9f97dec56bee4b3b6b953f47d27f3e332bf7b9e3148ed28bed9aa2dd6215c3",
      "bytes": 1158
    },
    {
      "file": "reviewer_training_quiz_answer_key_v1.json",
      "sha256": "a14cb81fec199888395b3db963af4827c0f86a97150cb6b3b01a5a52d3600ae5",
      "bytes": 6

In [9]:

# Cell 9 — Package Step 8 outputs

import shutil

ZIP_BASE = BASE_DIR / "NeuroFHIR_Review_WISH_Step08_Reviewer_Training_v1"
zip_path = shutil.make_archive(str(ZIP_BASE), "zip", OUT_DIR)

print("Packaged:", zip_path)
print("\nStep 8 COMPLETE when:")
print("  1) rubric wording is frozen,")
print("  2) validation is PASS,")
print("  3) the same version will be used for every reviewer,")
print("  4) case-specific expected answers remain hidden from reviewers.")


Packaged: /content/neurofhir_review_wish/NeuroFHIR_Review_WISH_Step08_Reviewer_Training_v1.zip

Step 8 COMPLETE when:
  1) rubric wording is frozen,
  2) validation is PASS,
  3) the same version will be used for every reviewer,
  4) case-specific expected answers remain hidden from reviewers.



# Stop point after this notebook

After this notebook passes.

The next blueprint item is **Step 9 — Recruit reviewer(s) and apply the recruitment decision gate**:

- **Path A:** neuro specialist confirmed.
- **Path B1:** no neuro specialist, but a clinical/informatics or domain-adjacent expert is available.
- **Path B2:** trained health-informatics students/researchers using the standardized rubric.

